# UMUD EXP59 auto Kaggle run: `seg59_02_highres_512_unet`

This is the no-edit notebook to import into Kaggle.

Run setup:
1. Import this notebook on Kaggle.
2. Add the competition input: **UMUD Challenge: Muscle Architecture in Ultrasound Data**.
3. Set **Accelerator = GPU** and **Internet = On**.
4. Run All.

What it does automatically:
- Downloads the current pipeline scripts from GitHub.
- Verifies `segment_then_measure.py` is pipeline version `2026-06-13.03`.
- Runs the EXP59 conservative high-resolution segmentation candidate:
  - `UMUD_EPOCHS=18`
  - `UMUD_IMG_SIZE=512`
  - `UMUD_MODEL_ARCH=unet`
  - `UMUD_MODEL_ENCODER=resnet34`
  - `UMUD_LOSS_MODE=dice_bce`
  - `UMUD_AUG_LEVEL=light`
  - `UMUD_CLAHE=0`
  - `UMUD_FASC_POS_WEIGHT=0`
  - `UMUD_WEIGHTS_TAG=seg59_02`
- Writes and validates `/kaggle/working/submission_segmentation.csv`.
- Bundles the submission, debug CSV, run config, and trained weights into one zip for download.


In [ ]:
# Download dependencies and current repo scripts. Internet must be On.
import os
import subprocess
import sys
import urllib.request
from pathlib import Path

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'segmentation-models-pytorch', 'albumentations'])

RAW_BASE = 'https://raw.githubusercontent.com/LacrimaeAware/kaggle-fun/main/umud-muscle-architecture'
SCRIPTS = [
    'segment_then_measure.py',
    'tick_calibration.py',
    'scale_ticks.py',
    'subpixel_spacing.py',
]
for script in SCRIPTS:
    url = f'{RAW_BASE}/{script}'
    urllib.request.urlretrieve(url, script)
    assert Path(script).exists(), f'{script} failed to download; check Internet = On'

EXPECTED_VERSION = '2026-06-13.03'
got = 'MISSING'
for line in Path('segment_then_measure.py').read_text(encoding='utf-8').splitlines():
    if line.startswith('PIPELINE_VERSION'):
        got = line.split('"')[1]
        break
print(f'>>> downloaded pipeline VERSION: {got}   (expected {EXPECTED_VERSION}) <<<')
assert got == EXPECTED_VERSION, 'STALE script: GitHub raw served an old file. Wait 1-2 minutes and re-run this cell.'

import torch
print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'no GPU')


In [ ]:
# Fixed EXP59 candidate config: no edits required.
import json, os, platform, sys
from pathlib import Path

RUN_ID = 'seg59_02_highres_512_unet'
CONFIG = {
    'UMUD_EPOCHS': '18',
    'UMUD_IMG_SIZE': '512',
    'UMUD_MODEL_ARCH': 'unet',
    'UMUD_MODEL_ENCODER': 'resnet34',
    'UMUD_MODEL_ENCODER_WEIGHTS': 'imagenet',
    'UMUD_LOSS_MODE': 'dice_bce',
    'UMUD_AUG_LEVEL': 'light',
    'UMUD_CLAHE': '0',
    'UMUD_FASC_POS_WEIGHT': '0',
    'UMUD_WEIGHTS_TAG': 'seg59_02',
    # Keep production defaults explicit so the log is not ambiguous.
    'UMUD_SCALE_ROUTER': '1',
    'UMUD_TTA': '1',
    'UMUD_FL_IDENTITY_BLEND': '0',
    'UMUD_FRAGMENT_FL': '1',
    'UMUD_USE_IDENTITY_FL': '1',
    'UMUD_FL_RECENTER': '1',
    'UMUD_TOP_BOUNDARY_MODE': 'line',
    'UMUD_MT_MODE': 'perp_center',
}
for key, value in CONFIG.items():
    os.environ[key] = value

run_config = {
    'run_id': RUN_ID,
    'config': CONFIG,
    'python': sys.version,
    'platform': platform.platform(),
}
Path('run_config_seg59_02.json').write_text(json.dumps(run_config, indent=2), encoding='utf-8')
print('Selected run:', RUN_ID)
for key in sorted(CONFIG):
    print(f'{key}={os.environ[key]}')


In [ ]:
# Full train + inference run. This produces submission_segmentation.csv and tagged weights.
import subprocess
import sys
subprocess.check_call([sys.executable, 'segment_then_measure.py'])


In [ ]:
# Validate the submission and debug outputs before download.
import os
from pathlib import Path
import pandas as pd

sub_path = Path('/kaggle/working/submission_segmentation.csv')
dbg_path = Path('/kaggle/working/calibration_measurement_debug.csv')
assert sub_path.exists(), 'missing submission_segmentation.csv'
assert dbg_path.exists(), 'missing calibration_measurement_debug.csv'

sub = pd.read_csv(sub_path)
expected_cols = ['image_id', 'pa_deg', 'fl_mm', 'mt_mm']
print('submission shape:', sub.shape)
print(sub.describe(include='all'))
assert list(sub.columns) == expected_cols, f'bad columns: {list(sub.columns)}'
assert sub.shape == (309, 4), f'expected 309 rows x 4 cols, got {sub.shape}'
assert sub['image_id'].is_unique, 'duplicate image_id rows'
assert not sub[expected_cols[1:]].isna().any().any(), 'NaN in numeric predictions'
assert sub['pa_deg'].between(5, 45).all(), 'PA outside expected clipped range'
assert sub['fl_mm'].between(20, 140).all(), 'FL outside expected clipped range'
assert sub['mt_mm'].between(5, 40).all(), 'MT outside expected clipped range'
assert sub['mt_mm'].std() > 1.0, 'MT looks constant; scale/measurement did not run'
assert sub['fl_mm'].std() > 1.0, 'FL looks constant; geometry/scale did not run'

dbg = pd.read_csv(dbg_path)
n_scaled = int((dbg['calibration_method'] != 'none').sum()) if 'calibration_method' in dbg else 0
print(f'calibrated images: {n_scaled}/309')
if 'calibration_method' in dbg:
    print(dbg['calibration_method'].value_counts(dropna=False).head(20).to_string())
assert n_scaled > 150, f'only {n_scaled} images scaled; expected the router to fire on most images'

weights = sorted(Path('/kaggle/working').glob('seg_*.pt'))
print('weights:', [w.name for w in weights])
assert any(w.name == 'seg_apo_seg59_02.pt' for w in weights), 'missing seg_apo_seg59_02.pt'
assert any(w.name == 'seg_fasc_seg59_02.pt' for w in weights), 'missing seg_fasc_seg59_02.pt'
print('OK: submission/debug/weights passed sanity checks.')


In [ ]:
# Bundle everything useful into one zip and show download links.
from pathlib import Path
from zipfile import ZipFile, ZIP_DEFLATED
from IPython.display import FileLink, display

work = Path('/kaggle/working')
zip_path = work / 'umud_seg59_02_highres_512_unet_outputs.zip'
include_names = [
    'submission_segmentation.csv',
    'calibration_measurement_debug.csv',
    'run_config_seg59_02.json',
]
include_paths = [work / name for name in include_names]
include_paths += sorted(work.glob('seg_*.pt'))
include_paths += sorted(work.glob('*.log'))
include_paths = [p for p in include_paths if p.exists()]

with ZipFile(zip_path, 'w', compression=ZIP_DEFLATED) as zf:
    for path in include_paths:
        zf.write(path, arcname=path.name)

print('Created:', zip_path)
print('Contents:')
for path in include_paths:
    print(' -', path.name, f'({path.stat().st_size:,} bytes)')

display(FileLink(str(zip_path)))
for path in include_paths:
    if path.suffix.lower() in {'.csv', '.json', '.pt'}:
        display(FileLink(str(path)))
